## Load Data and Split

In [ ]:
# Importing all the necessary libraries and modules
import os, sys
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
import random
from pathlib import Path
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from torchvision import datasets
from torch.utils.data import DataLoader

In [ ]:
# project root directory
project_root = "/Users/cindychen/Desktop/EE562/EE562 Assignments/EE562-Classifiers"
if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
#importing the ResNet-18 classifier and the dataloader functions from the doggie_loader.py file
from src.datasets.doggie_loader import get_doggie_dataset, create_doggie_dataloaders, get_default_transforms

## ResNet-18 CNN

Importing all the functions from the resnet file

In [ ]:
# importing the functions from the resnet module
from src.models.resnet_cnn import (
    create_resnet18,
    get_default_transforms,
    train_epoch,
    validate_epoch,
    predict_image,
    get_all_predictions
)

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)


In [ ]:
#Load dataset

train_dataset, val_dataset, test_dataset, class_names, num_classes = get_doggie_dataset()

train_loader, val_loader, test_loader = create_doggie_dataloaders(
    train_dataset, 
    val_dataset, 
    test_dataset, 
    batch_size=16, 
    num_workers=0
)

print(f"Number of classes: {num_classes}")
print(f"Class names: {class_names[:5]}...")

In [ ]:
# start the model
model = create_resnet18(num_classes=num_classes)

# model parameter count
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"trainable parameters: {trainable_params:,}")

In [ ]:
#  Training setup
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.1, patience=3
)

print(f"Loss function: CrossEntropyLoss")
print(f"Optimizer: Adam (lr=0.0001, weight_decay=1e-4)")
print(f"Scheduler: ReduceLROnPlateau")

In [ ]:
#training loop
num_epochs = 30
best_val_acc = 0.0

# Lists to store metrics
train_losses, val_losses = [], []
train_accs, val_accs = [], []
#training loop with validation and model saving
for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer) #training for one epoch and getting the training loss and accuracy
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    
    val_loss, val_acc = validate_epoch(model, val_loader, criterion) #validation for one epoch and getting the validation loss and accuracy
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    
    scheduler.step(val_loss) # step the scheduler based on the validation loss
    
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_dog_model.pth')
        print(f"Best model: (val_acc: {val_acc:.2f}%)")

In [ ]:
#plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# plot 1: loss
ax1.plot(train_losses, 'o-', color='pink', label='Train Loss', markersize=4, markerfacecolor='pink', markeredgecolor='pink')
ax1.plot(val_losses, 'o-', color='green', label='Val Loss', markersize=4, markerfacecolor='green', markeredgecolor='green')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True)

# plot 2 accuracy
ax2.plot(train_accs, 'o-', color='pink', label='Train Accuracy', markersize=4, markerfacecolor='pink', markeredgecolor='pink')
ax2.plot(val_accs, 'o-', color='green', label='Val Accuracy', markersize=4, markerfacecolor='green', markeredgecolor='green')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training and Validation Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

In [ ]:
#test evaluation

model.load_state_dict(torch.load('best_dog_model.pth'))
test_loss, test_acc = validate_epoch(model, test_loader, criterion)
print(f"\nFinal Test Results:")
print(f"  Test Loss: {test_loss:.4f}")
print(f"  Test Accuracy: {test_acc:.2f}%")

In [ ]:
#confusion matrix
test_preds, test_labels = get_all_predictions(model, test_loader)
cm = confusion_matrix(test_labels, test_preds)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=False, cmap='Blues', cbar=True)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

class_acc = cm.diagonal() / cm.sum(axis=1)
print(f"Mean class accuracy: {np.nanmean(class_acc):.4f}")

In [ ]:
#prediction of a single image
test_idx = random.randint(0, len(test_dataset) - 1)
img_path, true_label = test_dataset.samples[test_idx]
true_breed = class_names[true_label]

pred_breed, confidence = predict_image(img_path, model, class_names, val_transform)
clean_true = ' '.join(true_breed.split('-')[1].split('_')) if '-' in true_breed else true_breed
clean_pred = ' '.join(pred_breed.split('-')[1].split('_')) if '-' in pred_breed else pred_breed

print(f"True breed: {clean_true}")
print(f"Predicted: {clean_pred}")
print(f"Confidence: {confidence:.4f} ({confidence*100:.2f}%)")

In [ ]:
from PIL import Image

# Find best and worst performing breeds (ignoring any NaN values)
valid_indices = ~np.isnan(class_acc)
valid_class_acc = class_acc[valid_indices]
valid_class_names = [name for i, name in enumerate(class_names) if valid_indices[i]]

if len(valid_class_acc) > 0:
    best_idx_in_valid = np.argmax(valid_class_acc)
    worst_idx_in_valid = np.argmin(valid_class_acc)
    
    # get the original indices
    best_breed_idx = [i for i, valid in enumerate(valid_indices) if valid][best_idx_in_valid]
    worst_breed_idx = [i for i, valid in enumerate(valid_indices) if valid][worst_idx_in_valid]
    
    best_breed = class_names[best_breed_idx]
    worst_breed = class_names[worst_breed_idx]
    
    
    print(f"Best performing breed: {best_breed}")
    print(f"Accuracy: {class_acc[best_breed_idx]*100:.2f}%")
    
    
    print(f"Worst performing breed: {worst_breed}")
    print(f"Accuracy: {class_acc[worst_breed_idx]*100:.2f}%")
   
    
    # Show examples of best breed
    print(f"Examples of {best_breed} (Best Performing)")
    best_breed_indices = [i for i, (_, label) in enumerate(test_dataset.samples) 
                          if class_names[label] == best_breed]
    
    if best_breed_indices:
        # show up to 3 examples
        num_examples = min(3, len(best_breed_indices))
        fig, axes = plt.subplots(1, num_examples, figsize=(15, 5))
        if num_examples == 1:
            axes = [axes]
        
        for i, idx in enumerate(random.sample(best_breed_indices, num_examples)):
            img_path, true_label = test_dataset.samples[idx]
            img = Image.open(img_path)
            
            # get prediction
            pred_breed, confidence = predict_image(img_path, model, class_names, val_transform)
            
            axes[i].imshow(img)
            axes[i].axis('off')
            
            # clean breed names for display
            clean_true = ' '.join(best_breed.split('-')[1].split('_')) if '-' in best_breed else best_breed
            clean_pred = ' '.join(pred_breed.split('-')[1].split('_')) if '-' in pred_breed else pred_breed
            
            title = f"True: {clean_true}\nPred: {clean_pred}\nConf: {confidence*100:.1f}%"
            axes[i].set_title(title, fontsize=10)
        
        plt.suptitle(f"Best Breed: {clean_true} ({class_acc[best_breed_idx]*100:.1f}% accuracy)", fontsize=14)
        plt.tight_layout()
        plt.show()
    
    # Show examples of worst breed
    print(f" Examples of {worst_breed} (Worst Performing)")
    worst_breed_indices = [i for i, (_, label) in enumerate(test_dataset.samples) 
                           if class_names[label] == worst_breed]
    
    if worst_breed_indices:
        # Show up to 3 examples
        num_examples = min(3, len(worst_breed_indices))
        fig, axes = plt.subplots(1, num_examples, figsize=(15, 5))
        if num_examples == 1:
            axes = [axes]
        
        for i, idx in enumerate(random.sample(worst_breed_indices, num_examples)):
            img_path, true_label = test_dataset.samples[idx]
            img = Image.open(img_path)
            
            # Get prediction
            pred_breed, confidence = predict_image(img_path, model, class_names, val_transform)
            
            axes[i].imshow(img)
            axes[i].axis('off')
            
            # Clean breed names for display
            clean_true = ' '.join(worst_breed.split('-')[1].split('_')) if '-' in worst_breed else worst_breed
            clean_pred = ' '.join(pred_breed.split('-')[1].split('_')) if '-' in pred_breed else pred_breed
            
            # Color code the title based on correctness
            if pred_breed == worst_breed:
                title = f"Correct\nTrue: {clean_true}\nConf: {confidence*100:.1f}%"
            else:
                title = f"Incorrect\nTrue: {clean_true}\nPred: {clean_pred}\nConf: {confidence*100:.1f}%"
            
            axes[i].set_title(title, fontsize=10, color='green' if pred_breed == worst_breed else 'red')
        
        plt.suptitle(f"Worst Breed: {clean_true} ({class_acc[worst_breed_idx]*100:.1f}% accuracy)", fontsize=14)
        plt.tight_layout()
        plt.show()

else:
    print("No valid class accuracies found")